In [ ]:
!pip install openai pandas numpy scikit-learn tqdm


In [ ]:
import openai
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm


In [ ]:
from google.colab import userdata
import openai
import os

# Fetch secrets from Colab
OPENAI_API_KEY = userdata.get("API_KEY")
OPENAI_BASE_URL = userdata.get("BASE_URL").strip('"') # Strip leading/trailing quotes

# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL

# Configure OpenAI client
openai.api_key = OPENAI_API_KEY
openai.base_url = OPENAI_BASE_URL

print("✅ OpenAI API Key and Base URL loaded from Colab Secrets")

✅ OpenAI API Key and Base URL loaded from Colab Secrets


In [ ]:
data = [
    {"id": 1, "title": "AI beats humans at chess", "text": "An AI system defeated a grandmaster in a chess tournament."},
    {"id": 2, "title": "Stock market crash", "text": "Markets fell sharply due to economic uncertainty."},
    {"id": 3, "title": "New AI model released", "text": "A tech company released a new AI model for natural language processing."},
    {"id": 4, "title": "Football team wins championship", "text": "The local football team won the national championship."},
    {"id": 5, "title": "Machine learning in healthcare", "text": "ML is transforming medical diagnosis and treatment."}
]

df = pd.DataFrame(data)
df


,id,title,text
0,1,AI beats humans at chess,An AI system defeated a grandmaster in a chess...
1,2,Stock market crash,Markets fell sharply due to economic uncertainty.
2,3,New AI model released,A tech company released a new AI model for nat...
3,4,Football team wins championship,The local football team won the national champ...
4,5,Machine learning in healthcare,ML is transforming medical diagnosis and treat...


In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding


In [ ]:
embeddings = []

for text in tqdm(df["text"]):
    emb = get_embedding(text)
    embeddings.append(emb)

df["embedding"] = embeddings

100%|██████████| 5/5 [00:02<00:00,  2.36it/s]


In [ ]:
embedding_matrix = np.vstack(df["embedding"].values)
embedding_matrix.shape

(5, 1536)

In [ ]:
def recommend_similar_articles(article_index, top_k=3):
    query_embedding = embedding_matrix[article_index].reshape(1, -1)
    similarities = cosine_similarity(query_embedding, embedding_matrix)[0]

    similar_indices = similarities.argsort()[::-1][1:top_k+1]

    results = df.iloc[similar_indices][["title", "text"]]
    scores = similarities[similar_indices]

    return results.assign(similarity_score=scores)


In [ ]:
# Pick article 0: "AI beats humans at chess"
recommendations = recommend_similar_articles(article_index=0, top_k=3)
recommendations

,title,text,similarity_score
2,New AI model released,A tech company released a new AI model for nat...,0.382825
3,Football team wins championship,The local football team won the national champ...,0.261603
4,Machine learning in healthcare,ML is transforming medical diagnosis and treat...,0.202556
